# Marketing Attribution — Channel Performance
## Business Questions
- How much does each channel cost to acquire a customer (CAC)?
- Which channel generates the highest ROAS?
- How has spend evolved and how efficient is each channel over time?

**Channels:** Google Ads · Facebook Ads · SEO · Email
**Period:** Jan 2022 – Dec 2023 (24 months)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.float_format", "{:.2f}".format)
sns.set_theme(style="whitegrid", font_scale=1.1)

ROOT = Path("..")
IMG  = ROOT / "images"

spend = pd.read_csv(ROOT / "data" / "marketing_spend.csv")
acq   = pd.read_csv(ROOT / "data" / "customer_acquisitions.csv")
ltv   = pd.read_csv(ROOT / "data" / "cohort_ltv.csv")

COLORS = {
    "google_ads":   "#2a9d8f",
    "facebook_ads": "#e76f51",
    "seo":          "#264653",
    "email":        "#e9c46a",
}
CHANNEL_LABELS = {
    "google_ads": "Google Ads",
    "facebook_ads": "Facebook Ads",
    "seo": "SEO",
    "email": "Email",
}
print("Data loaded ✓")
print(f"  Spend rows   : {len(spend):,}")
print(f"  Acq rows     : {len(acq):,}")
print(f"  LTV rows     : {len(ltv):,}")

## 1 · Total Spend by Channel

In [ ]:
spend_by_ch = spend.groupby("channel")["spend"].sum().sort_values(ascending=False)
spend_by_ch.index = [CHANNEL_LABELS[c] for c in spend_by_ch.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

spend_by_ch.plot(kind="bar", ax=axes[0],
                 color=[COLORS["google_ads"], COLORS["facebook_ads"],
                        COLORS["seo"], COLORS["email"]],
                 edgecolor="white")
axes[0].set_title("Total 24-Month Spend by Channel")
axes[0].set_ylabel("Spend ($)")
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f"${v/1000:.0f}K"))
axes[0].tick_params(axis="x", rotation=30)

spend_by_ch.plot(kind="pie", ax=axes[1], autopct="%1.1f%%",
                 colors=[COLORS["google_ads"], COLORS["facebook_ads"],
                         COLORS["seo"], COLORS["email"]],
                 startangle=90, wedgeprops=dict(edgecolor="white", linewidth=1.5))
axes[1].set_title("Budget Share")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig(IMG / "01_spend_by_channel.png", dpi=150, bbox_inches="tight")
plt.show()

## 2 · CAC Trend by Channel

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for ch in acq["channel"].unique():
    d = acq[acq["channel"]==ch].copy()
    ax.plot(d["month"], d["cac"], label=CHANNEL_LABELS[ch],
            color=COLORS[ch], linewidth=2, marker="o", markersize=3)

ax.set_title("Monthly CAC by Channel (lower = better)")
ax.set_ylabel("CAC ($)")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(IMG / "02_cac_trend.png", dpi=150, bbox_inches="tight")
plt.show()

print("24-month average CAC:")
for ch, grp in acq.groupby("channel"):
    print(f"  {CHANNEL_LABELS[ch]:<14}: ${grp['cac'].mean():.2f}")

## 3 · ROAS — Return on Ad Spend

In [ ]:
# ROAS = Revenue / Spend. Revenue ≈ new_customers × ARPU (month 0)
ARPU = {"google_ads": 48, "facebook_ads": 39, "seo": 55, "email": 43}
acq["revenue"] = acq.apply(lambda r: r["new_customers"] * ARPU[r["channel"]], axis=1)
acq["roas"]    = acq["revenue"] / acq["spend"]

roas_avg = acq.groupby("channel")["roas"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors_ordered = [COLORS[c] for c in roas_avg.index]
bars = ax.barh([CHANNEL_LABELS[c] for c in roas_avg.index],
               roas_avg.values, color=colors_ordered, edgecolor="white")
ax.axvline(x=3, color="gray", linestyle="--", label="Minimum ROAS 3x")
ax.set_xlabel("ROAS (Revenue / Spend)")
ax.set_title("Average ROAS by Channel")
ax.legend()
for bar, v in zip(bars, roas_avg.values):
    ax.text(v + 0.05, bar.get_y() + bar.get_height()/2, f"{v:.1f}x",
            va="center", fontweight="bold")
plt.tight_layout()
plt.savefig(IMG / "03_roas_by_channel.png", dpi=150, bbox_inches="tight")
plt.show()

## 4 · Efficiency Matrix — CAC vs New Customers

In [ ]:
summary = acq.groupby("channel").agg(
    avg_cac=("cac","mean"),
    avg_new_customers=("new_customers","mean"),
    total_spend=("spend","sum"),
).reset_index()

fig, ax = plt.subplots(figsize=(9, 6))
for _, row in summary.iterrows():
    ax.scatter(row["avg_cac"], row["avg_new_customers"],
               s=row["total_spend"]/150, color=COLORS[row["channel"]],
               alpha=0.85, edgecolors="white", linewidth=1.5)
    ax.annotate(CHANNEL_LABELS[row["channel"]],
                (row["avg_cac"], row["avg_new_customers"]),
                xytext=(8, 5), textcoords="offset points", fontsize=10)

ax.set_xlabel("Average CAC ($)  ← lower is better")
ax.set_ylabel("Avg Monthly New Customers  ↑ higher is better")
ax.set_title("Efficiency Matrix: CAC vs Volume  (bubble size = total spend)")
ax.axvline(50, color="gray", linestyle=":", alpha=0.5)
ax.text(51, ax.get_ylim()[0]+5, "CAC = $50", fontsize=8, color="gray")
plt.tight_layout()
plt.savefig(IMG / "04_efficiency_matrix.png", dpi=150, bbox_inches="tight")
plt.show()